# 🔎 04. Embeddingi i topic modeling

**Cel:** porównać reprezentacje semantyczne i ocenić, czy klastry nadają się do interpretacji. Embeddingi otrzymują zachowaną kolumnę `document`. Lematyzacja służy opisowi tematów. Klaster grupuje dokumenty, a reprezentacja tematu opisuje tę grupę.

# 🧭 Jak wykonać ten notebook

1. Otwórz plik `.ipynb` w Google Colab i zapisz własną kopię na Dysku Google.
2. **Wystarczy CPU.** Komórka to jeden blok tekstu albo kodu. Kod uruchamiasz przyciskiem ▶ po lewej lub Shift+Enter.
3. Uruchom pierwszą komórkę kodu. Jeżeli zainstaluje biblioteki i pokaże 🔄, uruchom ponownie sesję z menu **Środowisko wykonawcze**. Potem zacznij od pierwszej komórki. Restart zachowuje pliki, ale usuwa zmienne z pamięci.
4. Wykonuj kod **od góry, bez pomijania komórek**. Dane pobiorą się automatycznie z [repozytorium prowadzącego](https://github.com/bartlomiejnowak-ux/PSPS-2026). Domyślnie niczego nie wgrywasz. W razie awarii pobierania komórka podaje instrukcję ręcznego wgrania.
5. Poczekaj, aż obracający się znacznik przy komórce zniknie. Pierwsze pobranie modelu i obliczenia mogą potrwać kilka minut, a BERTopic dłużej. Nie klikaj wielokrotnie ▶.
6. ✅ oznacza sukces, 🔎 wskazuje co przeczytać, ⚠️ ważne ograniczenie, ✏️ ćwiczenie. Emoji nie zmieniają działania kodu.
7. Na końcu pobierz ZIP wyników. Sam zapis notebooka na Dysku nie zachowuje plików z tymczasowej sesji.


### 🛠️ Gdy coś nie działa

| Objaw | Co zrobić |
|---|---|
| `NameError` lub „nie zdefiniowano” | Pominięto wcześniejszy krok albo zrestartowano sesję. Wykonaj kod od początku. |
| `ModuleNotFoundError` / błąd wersji biblioteki | Uruchom instalację, zrestartuj sesję i wykonaj komórki od góry. |
| Brak pliku / zła kolumna | Ponów komórkę pobierania danych. Awaryjnie wybierz `cleaned_topic_modeling_dataset(1).csv` z repozytorium. |
| Błąd pobierania modelu | Sprawdź połączenie, zaczekaj i ponów komórkę pobierania. Nie zmieniaj nazwy modelu. |
| Błąd po zmianie parametru | Cofnij zmianę lub przywróć wartości pokazane w komentarzach i wykonaj dalsze komórki kolejno. |
| Sesja wygasła | Połącz ponownie, uruchom notebook od początku i ponownie wykonaj komórkę pobierania danych. |

Nie przechodź dalej po czerwonym błędzie. Czytaj ostatnią linijkę komunikatu. W tej kopii wyniki pojawią się dopiero po uruchomieniu kodu. Liczby na slajdach pochodzą ze sprawdzonego wcześniejszego wykonania; przy innych ustawieniach lub środowisku wynik może się różnić.

⏳ **Czas na CPU:** samo tworzenie embeddingów może trwać około 10 minut. Pasek `Batches` pokazuje postęp partii dokumentów. Pozostaw kartę otwartą i poczekaj; nie uruchamiaj tej samej komórki ponownie, gdy obliczenia jeszcze trwają.

## 📖 Słowa i skróty używane w tym module

- **Embedding / wektor:** lista liczb opisująca tekst według modelu.
- **Wymiar:** jedna współrzędna takiej listy liczb.
- **MiniLM:** gotowy model zamieniający tekst w 384 liczby.
- **PCA:** zmniejszenie liczby wymiarów z zachowaniem możliwie dużej zmienności.
- **UMAP:** zmniejszenie liczby wymiarów przez przybliżenie sąsiedztwa punktów.
- **Klasteryzacja / HDBSCAN:** grupowanie podobnych tekstów / algorytm szukający gęstych skupień.
- **Szum / outliers / Topic −1:** teksty, których model nie przypisał do klastra.
- **Inliers:** teksty przypisane do klastrów.
- **Cosine similarity:** podobieństwo kierunków wektorów; większe oznacza bardziej zgodne kierunki.
- **Odległość euklidesowa:** odległość w linii prostej między punktami.
- **c-TF-IDF:** wybór słów częstych w temacie i wyróżniających go na tle innych.
- **MMR:** wybór słów trafnych, ale niepowtarzających ciągle tej samej informacji.
- **Seed / random_state:** liczba ustalająca przebieg losowania dla odtwarzalności.
- **Cache:** zapis wcześniejszych obliczeń do ponownego użycia.
- **Coherence:** miara współwystępowania słów opisujących temat.
- **Diversity:** udział różnych słów w opisach tematów.
- **NMI / ARI:** miary zgodności dwóch podziałów; ARI uwzględnia zgodność przypadkową.
- **Sensitivity:** sprawdzenie, jak wyniki zmieniają się po zmianie ustawień.
- **Vectorizer:** narzędzie zliczające słowa lub n-gramy.
- **full SVD:** pełna metoda obliczenia PCA, bez losowego przybliżenia.
- **Monotoniczność:** zmiana cały czas w jednym kierunku.
- **Graf sąsiadów:** zapis, które punkty są do siebie najbardziej podobne.
- **Wariancja:** miara rozrzutu danych.
- **Reduktor:** używana metoda zmniejszenia liczby wymiarów.

## 🔎 Kontrolowane porównanie

MiniLM → PCA → HDBSCAN oraz MiniLM → UMAP → HDBSCAN. Zmieniamy redukcję wymiarowości, zachowując embeddingi, parametry klasteryzacji i opis tematów. **Target nigdy nie służy do dopasowania modelu.** NMI/ARI porównują wyniki z etykietami dopiero po dopasowaniu.

# 🔎 1. Przygotowanie
Uruchom komórkę instalacji poniżej. W Colab wszystko wykonujesz w przeglądarce.

### ▶️ Krok kodu 1

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** ✅ Biblioteki gotowe albo instrukcja jednorazowego restartu 🔄.

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata as metadata
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
PACKAGES = ['pandas==3.0.5', 'numpy==2.5.3', 'matplotlib==3.11.2', 'scikit-learn==1.9.1', 'sentence-transformers==6.0.1', 'umap-learn==0.5.12', 'hdbscan==0.8.44', 'bertopic==0.17.4', 'spacy==3.8.16', 'gensim==4.4.0', 'plotly==7.0.0', 'nbformat==5.11.1', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl', 'scipy==1.18.1', 'torch==2.14.0', 'transformers==5.17.0']
if sys.version_info < (3, 12):
    raise RuntimeError('⛔ Ten zestaw wersji wymaga Python 3.12 lub nowszego. Wybierz zgodne środowisko.')
def installed(spec):
    name, expected = ('en-core-web-sm', '3.8.0') if spec.startswith('https:') else spec.split('==')
    try: return metadata.version(name) == expected
    except metadata.PackageNotFoundError: return False
needed = [spec for spec in PACKAGES if not installed(spec)]
# Biblioteki obrazu i dźwięku z bazowej sesji mogą być niezgodne z nowym torch.
# Ten warsztat analizuje tekst i ich nie używa.
optional_media = [name for name in ('torchvision', 'torchaudio') if IN_COLAB and importlib.util.find_spec(name) is not None]
if IN_COLAB and (needed or optional_media):
    if optional_media:
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', *optional_media])
    print('⏳ Instalacja bibliotek. Poczekaj na zakończenie tej komórki.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
    raise RuntimeError('🔄 Instalacja zakończona. Uruchom ponownie sesję z menu Środowisko wykonawcze, a następnie wykonaj komórki od początku. To jednorazowy krok po instalacji.')
if needed:
    raise RuntimeError('⛔ Brakuje wymaganych wersji. Lokalnie użyj pliku requirements właściwego modułu. W Colab komórka instaluje je sama. Braki: ' + ', '.join(needed))
print('✅ Biblioteki gotowe. Możesz uruchomić następną komórkę.')


### ▶️ Krok kodu 2

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
from pathlib import Path
from itertools import combinations
import warnings
import os
import json
import hashlib
import time
import platform
from importlib.metadata import version
from importlib.util import find_spec
RUN_STARTED = time.perf_counter()
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
WORKSHOP_ROOT = BASE_DIR.parent if BASE_DIR.name == "04_EMBEDDINGS_TOPIC_MODELING" else BASE_DIR
BASE_DIR = WORKSHOP_ROOT / "04_EMBEDDINGS_TOPIC_MODELING"
BASE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HOME", str(WORKSHOP_ROOT / "cache" / "huggingface"))
import torch
torch.set_num_threads(min(4, os.cpu_count() or 1))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP
from hdbscan import HDBSCAN

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance
from sklearn.feature_extraction.text import CountVectorizer

import spacy
from spacy.lang.en.stop_words import STOP_WORDS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Narzędzia wczytane.")


# 🔎 2. Wczytanie danych

Wczytujemy ten sam korpus co w modułach 1–3. Kody: 0 Stress, 1 Depression, 2 Bipolar, 3 Personality Disorder, 4 Anxiety. Etykiety opisują grupy źródłowe dokumentów, nie potwierdzone diagnozy osób. Dostarczony plik jest już nazwany cleaned, a historia wcześniejszego czyszczenia pozostaje nieznana.

### ▶️ Krok kodu 3

Uruchom raz i poczekaj. **Oczekiwany efekt:** automatyczne pobranie danych i komunikat ✅. Przy awarii ustaw w kodzie `DATA_SOURCE = 'upload'` i wybierz **`cleaned_topic_modeling_dataset(1).csv`** z repozytorium prowadzącego.

In [ ]:
# 📥 Dane z repozytorium prowadzącego. Domyślnie niczego nie wgrywasz ręcznie.
import io, urllib.request, hashlib
DATA_SOURCE = 'github'  # awaryjnie zmień na 'upload' i uruchom komórkę ponownie
GITHUB_COMMIT = '1cb710169713a4f8ea5a0d839be6d6e1df8ab078'
GITHUB_URL = f'https://raw.githubusercontent.com/bartlomiejnowak-ux/PSPS-2026/{GITHUB_COMMIT}/cleaned_topic_modeling_dataset%281%29.csv'
EXPECTED_SHA256 = '4fb7bdc1779127e875ce3ea7fd571f85885b700937b9439bd606b7c2d67d7972'
INPUT_NAME = 'cleaned_topic_modeling_dataset(1).csv'
if IN_COLAB:
    if DATA_SOURCE == 'github':
        print('⏳ Pobieranie danych z GitHub…')
        try:
            with urllib.request.urlopen(GITHUB_URL, timeout=60) as response:
                data_bytes = response.read()
        except Exception as exc:
            raise RuntimeError('⛔ Nie udało się pobrać danych. Sprawdź internet i ponów tę komórkę. Awaryjnie ustaw DATA_SOURCE = "upload" i wybierz plik z repozytorium prowadzącego.') from exc
        if hashlib.sha256(data_bytes).hexdigest() != EXPECTED_SHA256:
            raise ValueError('⛔ Pobrany plik nie zgadza się ze sprawdzoną wersją. Nie kontynuuj analizy na tym pliku.')
    elif DATA_SOURCE == 'upload':
        from google.colab import files
        print('📂 Wybierz cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('⛔ Wybierz dokładnie jeden plik i ponów tę komórkę.')
        data_bytes = next(iter(uploaded.values()))
    else:
        raise ValueError('⛔ DATA_SOURCE musi mieć wartość "github" albo "upload".')
    try:
        source_df = pd.read_csv(io.BytesIO(data_bytes), keep_default_na=False)
    except Exception as exc:
        raise ValueError('⛔ Nie można odczytać pliku. Wgraj cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.') from exc
    missing_columns = {'target', 'document'} - set(source_df.columns)
    if source_df.empty or missing_columns:
        raise ValueError(f'⛔ Pusty lub niewłaściwy plik. Brakujące kolumny: {sorted(missing_columns)}')
    data_folder = WORKSHOP_ROOT / 'data'
    data_folder.mkdir(parents=True, exist_ok=True)
    (data_folder / INPUT_NAME).write_bytes(data_bytes)
    print(f'✅ Dane gotowe: {len(source_df)} wierszy. Źródło: {DATA_SOURCE}.')


### ▶️ Krok kodu 4

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
def find_dataset(base_dir):
    # Dataset lives in the shared, relative data directory.
    for folder in [Path(base_dir)]:
        candidates = sorted(folder.glob("cleaned_topic_modeling_dataset*.csv"))
        if len(candidates) > 1:
            raise ValueError(f"Multiple datasets in {folder}: {candidates}. Keep one input CSV.")
        if candidates:
            return candidates[0]
    raise FileNotFoundError("Put cleaned_topic_modeling_dataset*.csv in the shared data directory; start from the module or WORKSHOP_FINAL directory.")

DATA_PATH = find_dataset(WORKSHOP_ROOT / "data")
df = pd.read_csv(DATA_PATH)
required = {"target", "document"}
if not required.issubset(df.columns):
    raise ValueError(f"Brak kolumn: {sorted(required - set(df.columns))}")
if df["document"].isna().any() or not df["document"].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
    raise ValueError("Missing, empty or non-string document. Resolve explicitly before fitting.")
if df["target"].isna().any() or not df["target"].isin(range(5)).all():
    raise ValueError("Expected external target codes 0 through 4 without missing values.")
df.insert(0, "source_row", np.arange(len(df)))
TARGET_NAMES = {0: "Stress", 1: "Depression", 2: "Bipolar", 3: "Personality Disorder", 4: "Anxiety"}
display(df.head())
print("Dataset:", DATA_PATH, "Shape:", df.shape)
print(df["target"].value_counts().sort_index().rename(index=TARGET_NAMES))
print("Exact duplicate texts:", df["document"].duplicated().sum(), "(retained; report as a corpus limitation)")


# 🔎 3. Kontrola danych

Sprawdziliśmy liczebności, puste teksty i duplikaty. Powtórzenia pozostają w analizie podstawowej. `source_row` identyfikuje wiersz źródłowego CSV, licząc od zera.

Opcjonalny `FAST_MODE` wybiera 1000 dokumentów z seed 42, bez stratyfikacji po target. To próba dydaktyczna, odrębna od pełnego wyniku. Domyślnie analizujemy cały korpus.

### ▶️ Krok kodu 5

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
FAST_MODE = os.environ.get("BERTOPIC_FAST_MODE", "0") == "1"
FAST_N = 1000
work_df = (df.sample(n=min(len(df), FAST_N), random_state=RANDOM_STATE).sort_index()
           if FAST_MODE else df).reset_index(drop=True)
texts = work_df["document"].tolist()
y_external = work_df["target"].to_numpy()
assert len(texts) == len(y_external) == len(work_df)
assert work_df["source_row"].is_unique
assert texts == df.iloc[work_df["source_row"].to_numpy()]["document"].tolist()
print(f"Documents used: {len(texts):,}; FAST_MODE={FAST_MODE}")


# 🔎 4. Analiza
## 🔎 4.1 Embeddingi MiniLM

`all-MiniLM-L6-v2` daje wektory o 384 wymiarach. Podobne znaczenia mogą mieć podobne wektory mimo różnych słów. Model ogranicza wejście do 256 tokenów modelu, więc długie dokumenty są obcinane. Token modelu różni się od tokena słownego w module 1.

### ▶️ Krok kodu 6

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# Pin the public model snapshot as well as the Python environment.
MODEL_REVISION = "c9745ed1d9f207416be6d2e6f8de32d1f16199bf"
embedding_model = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION, device="cpu")
assert embedding_model.get_sentence_embedding_dimension() == 384
CACHE_DIR = WORKSHOP_ROOT / "cache"
CACHE_DIR.mkdir(exist_ok=True)
cache_config = {"model": MODEL_NAME, "revision": MODEL_REVISION,
                "normalize_embeddings": True, "max_seq_length": embedding_model.max_seq_length,
                "sentence_transformers": version("sentence-transformers"),
                "transformers": version("transformers"), "torch": version("torch")}

def embedding_cache_key(documents, config):
    payload = json.dumps({"config": config, "documents": documents}, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

cache_key = embedding_cache_key(texts, cache_config)
EMB_PATH = CACHE_DIR / f"minilm_{cache_key}.npy"
cache_hit = EMB_PATH.exists()
if cache_hit:
    embeddings = np.load(EMB_PATH, allow_pickle=False)
else:
    embeddings = embedding_model.encode(texts, show_progress_bar=True, batch_size=64,
                                        normalize_embeddings=True, convert_to_numpy=True)
    np.save(EMB_PATH, embeddings, allow_pickle=False)
assert embeddings.shape == (len(texts), 384), "Cache shape mismatch"
assert np.isfinite(embeddings).all(), "Non-finite embedding"
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)
embedding_digest = hashlib.sha256(embeddings.tobytes()).hexdigest()
print("Loaded cache:" if cache_hit else "Saved cache:", EMB_PATH.name)
print("Embedding matrix:", embeddings.shape)
print("Long documents are truncated at", embedding_model.max_seq_length, "model tokens.")


## 🔎 4.2 Cosine similarity

Porównujemy trzy skonstruowane zdania. Cosine similarity opisuje kierunki wektorów, a nie liczbę wspólnych słów. Wynik VADER i podobieństwo semantyczne odpowiadają na różne pytania.

### ▶️ Krok kodu 7

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
demo_sentences = [
    "I feel anxious before an exam.",
    "Exams make me nervous and worried.",
    "I bought a new bicycle yesterday.",
]
demo_emb = embedding_model.encode(demo_sentences, normalize_embeddings=True)
demo_sim = pd.DataFrame(
    cosine_similarity(demo_emb),
    index=demo_sentences,
    columns=demo_sentences,
)
display(demo_sim.round(3))


## 🔎 4.3 Redukcja wymiarowości

PCA i UMAP zmniejszają liczbę wymiarów. HDBSCAN wyznacza klastry w uzyskanej przestrzeni.

PCA zachowuje kierunki największej wariancji i przy `svd_solver="full"` jest deterministycznym punktem odniesienia. UMAP przybliża lokalne sąsiedztwa, zależy od parametrów i seed. Obie redukcje mogą zmienić strukturę gęstości.

### ▶️ Krok kodu 8

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
N_COMPONENTS = 10

pca_reducer = PCA(n_components=N_COMPONENTS, svd_solver="full")
umap_reducer = UMAP(
    n_neighbors=15,
    n_components=N_COMPONENTS,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

pca_10 = pca_reducer.fit_transform(embeddings)
umap_10 = umap_reducer.fit_transform(embeddings)

print("PCA reduced shape:", pca_10.shape)
print("UMAP reduced shape:", umap_10.shape)
print("PCA explained variance (10 components):", round(pca_reducer.explained_variance_ratio_.sum(), 3))


## 🔎 4.4 HDBSCAN i szum

`min_cluster_size=30` określa minimalny rozmiar klastra, a `min_samples=7` wpływa na ocenę gęstości. Parametry są wspólne dla obu wariantów. Topic `-1` oznacza szum i nie jest merytorycznym tematem. Mniejszy min_cluster_size może dać bardziej szczegółowy podział, ale nie gwarantuje monotonicznej zmiany liczby tematów.

### ▶️ Krok kodu 9

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
HDBSCAN_PARAMS = dict(
    min_cluster_size=30,
    min_samples=7,
    metric="euclidean",
    prediction_data=True,
)

def fit_hdbscan(reduced):
    model = HDBSCAN(**HDBSCAN_PARAMS)
    labels = model.fit_predict(reduced)
    return model, labels

hdb_pca, labels_pca_raw = fit_hdbscan(pca_10)
hdb_umap, labels_umap_raw = fit_hdbscan(umap_10)

for name, labels in [("PCA", labels_pca_raw), ("UMAP", labels_umap_raw)]:
    n_topics = len(set(labels)) - (1 if -1 in labels else 0)
    outlier_rate = np.mean(labels == -1)
    print(f"{name}: {n_topics} clusters; outliers = {outlier_rate:.1%}")


## 🔎 4.5 Reprezentacja tematu: c-TF-IDF i MMR

HDBSCAN grupuje dokumenty. c-TF-IDF wskazuje terminy odróżniające klaster, a MMR ogranicza redundancję opisu.

Lematyzujemy dokumenty pojedynczo, zachowując oryginały do embeddingów i czytania. Oba modele otrzymują te same teksty reprezentacji i wspólny słownik unigramów/bigramów z `min_df=2` na poziomie dokumentów. Standardowo BERTopic stosuje ten próg do zagregowanych tekstów klastrów. Ta jawna decyzja zapewnia wspólny słownik porównania.

### ▶️ Krok kodu 10

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
CUSTOM_STOPWORDS = set(STOP_WORDS) | {"mental", "health", "reddit", "im", "dont", "ive", "id", "like"}

def lemmatized_document(doc):
    return " ".join(token.lemma_.lower() for token in doc
                    if not token.is_stop and not token.is_punct and not token.is_space
                    and token.lemma_.lower() not in CUSTOM_STOPWORDS)

lemma_config = {"spacy": version("spacy"), "model": nlp.meta.get("version"),
                "stopwords": sorted(CUSTOM_STOPWORDS), "procedure": "lemma-stop-punct-space-v1"}
lemma_path = CACHE_DIR / f"lemmas_{embedding_cache_key(texts, lemma_config)}.json"
if lemma_path.exists():
    representation_texts = json.loads(lemma_path.read_text(encoding="utf-8"))
else:
    representation_texts = [lemmatized_document(doc) for doc in nlp.pipe(texts, batch_size=64)]
    lemma_path.write_text(json.dumps(representation_texts, ensure_ascii=False), encoding="utf-8")
assert len(representation_texts) == len(texts)
assert all(isinstance(t, str) for t in representation_texts)
empty_representations = sum(not t.strip() for t in representation_texts)
print("Empty after lemmatization:", empty_representations,
      "(retained with original embeddings; contribute no topic words)")
VECTORIZER_PARAMS = dict(lowercase=False, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2))
vocabulary = CountVectorizer(**VECTORIZER_PARAMS, min_df=2).fit(representation_texts).vocabulary_

def make_vectorizer():
    return CountVectorizer(**VECTORIZER_PARAMS, vocabulary=dict(vocabulary))


## 🔎 4.6 Dopasowanie dwóch modeli

Zmieniamy wyłącznie reduktor. Każdy model ma świeże obiekty HDBSCAN, vectorizer, c-TF-IDF i MMR o identycznych ustawieniach. Asercje sprawdzają niezmienione embeddingi, słownik oraz zgodność klastrów z wcześniejszą demonstracją.

### ▶️ Krok kodu 11

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
def make_topic_model(reducer):
    # Fresh mutable objects, identical settings. Never share fitted vectorizers.
    return BERTopic(embedding_model=embedding_model, umap_model=reducer,
                    hdbscan_model=HDBSCAN(**HDBSCAN_PARAMS),
                    vectorizer_model=make_vectorizer(),
                    ctfidf_model=ClassTfidfTransformer(),
                    representation_model=MaximalMarginalRelevance(diversity=0.1),
                    calculate_probabilities=False, verbose=True)

pca_topic_model = make_topic_model(PCA(n_components=N_COMPONENTS, svd_solver="full"))
umap_topic_model = make_topic_model(UMAP(n_neighbors=15, n_components=N_COMPONENTS,
                                      min_dist=0.0, metric="cosine", random_state=RANDOM_STATE, n_jobs=1))
topics_pca, _ = pca_topic_model.fit_transform(representation_texts, embeddings=embeddings)
topics_umap, _ = umap_topic_model.fit_transform(representation_texts, embeddings=embeddings)
assert len(topics_pca) == len(topics_umap) == len(texts)
assert hashlib.sha256(embeddings.tobytes()).hexdigest() == embedding_digest
assert pca_topic_model.vectorizer_model is not umap_topic_model.vectorizer_model
assert pca_topic_model.vectorizer_model.vocabulary_ == umap_topic_model.vectorizer_model.vocabulary_
# Reducer demonstrations and actual BERTopic fits must agree (cluster IDs can differ).
assert adjusted_rand_score(labels_pca_raw, topics_pca) == 1.0
assert adjusted_rand_score(labels_umap_raw, topics_umap) == 1.0
print("PCA model topics (including -1 noise):")
display(pca_topic_model.get_topic_info().head(12))
print("UMAP model topics (including -1 noise):")
display(umap_topic_model.get_topic_info().head(12))


## 🔎 4.7 Słowa tematów i dokumenty reprezentatywne

Czytamy słowa i oryginalne dokumenty razem. Nazwa tematu wymaga interpretacji. Dokumenty reprezentatywne opisują centrum rozwiązania i nie zastępują inspekcji przypadków granicznych oraz szumu.

### ▶️ Krok kodu 12

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
def topic_overview(model, n_topics=8, n_words=10):
    info = model.get_topic_info()
    topic_ids = [t for t in info["Topic"].tolist() if t != -1][:n_topics]
    rows = []
    for tid in topic_ids:
        words = [w for w, _ in model.get_topic(tid)[:n_words]]
        rows.append({"topic": tid, "top_words": ", ".join(words)})
    return pd.DataFrame(rows)

print("PCA + HDBSCAN")
display(topic_overview(pca_topic_model))
print("UMAP + HDBSCAN")
display(topic_overview(umap_topic_model))

# Map representative lemmatized documents back to their original source rows.
def representative_rows(model, name):
    rows = []
    for tid in model.get_topic_info()["Topic"]:
        for representative in model.get_representative_docs(tid) or []:
            matches = [i for i, value in enumerate(representation_texts) if value == representative]
            for i in matches:
                rows.append({"model": name, "topic": tid, "source_row": int(work_df.iloc[i]["source_row"]),
                             "document": texts[i]})
    return pd.DataFrame(rows, columns=["model", "topic", "source_row", "document"])

representatives = pd.concat([representative_rows(pca_topic_model, "PCA"),
                             representative_rows(umap_topic_model, "UMAP")], ignore_index=True)
display(representatives.head(12))
print("Read these alongside topic words. Topic -1 is noise, not a substantive topic.")


## 🔎 Interaktywny podgląd słów

Wykres pokazuje tematy poza szumem. Zapiszemy go jako HTML do dalszej inspekcji.

### ▶️ Krok kodu 13

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
non_noise_topics = [tid for tid in umap_topic_model.get_topics() if tid != -1]
topic_barchart = None
if non_noise_topics:
    topic_barchart = umap_topic_model.visualize_barchart(topics=non_noise_topics[:10], n_words=8)
    display(topic_barchart)
else:
    print("All documents are noise: no non-noise topic barchart.")


## 🔎 4.8 Ocena według wielu kryteriów

Zestawiamy liczbę tematów, odsetek szumu, udział największego tematu, NMI/ARI i interpretowalność. `largest_topic_share_inliers` ma w mianowniku wyłącznie dokumenty poza szumem. NMI/ARI dla wszystkich wierszy traktują `-1` jako jedną etykietę. Wariant inliers ma mniejsze i różne pokrycie, więc nie jest prostym zamiennikiem.

### ▶️ Krok kodu 14

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
def summarize_assignments(name, topics, y):
    topics = np.asarray(topics)
    y = np.asarray(y)
    inlier = topics != -1
    topic_ids, counts = np.unique(topics[inlier], return_counts=True)
    return {
        "model": name,
        "n_topics": len(topic_ids),
        "outlier_rate": np.mean(~inlier),
        "largest_topic_share_inliers": counts.max() / counts.sum() if len(counts) else np.nan,
        "NMI_all": normalized_mutual_info_score(y, topics),
        "ARI_all": adjusted_rand_score(y, topics),
        "NMI_inliers": normalized_mutual_info_score(y[inlier], topics[inlier]) if inlier.sum() >= 2 else np.nan,
        "ARI_inliers": adjusted_rand_score(y[inlier], topics[inlier]) if inlier.sum() >= 2 else np.nan,
    }

comparison = pd.DataFrame([
    summarize_assignments("PCA + HDBSCAN", topics_pca, y_external),
    summarize_assignments("UMAP + HDBSCAN", topics_umap, y_external),
])
display(comparison.round(3))


## 🔎 4.9 Coherence i topic diversity

Coherence c_v opisuje współwystępowanie terminów. Wspólny korpus ewaluacyjny zawiera uporządkowane unigramy i bigramy, aby obejmował wszystkie cechy vectorizera. To wybór zależny od reprezentacji, nie uniwersalna miara trafności.

Topic diversity to udział różnych terminów wśród terminów z pierwszych 10 pozycji tematów. Pomijamy Topic -1. Wysoka wartość nie dowodzi psychologicznej trafności ani dobrego pokrycia dokumentów.

### ▶️ Krok kodu 15

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

def coherence_features(document):
    tokens = make_vectorizer().build_tokenizer()(document)
    features = []
    for i, token in enumerate(tokens):
        features.append(token)
        if i + 1 < len(tokens):
            features.append(token + " " + tokens[i + 1])
    return features

tokenized_corpus = [coherence_features(document) for document in representation_texts]
coherence_dictionary = Dictionary(tokenized_corpus)

def topic_scores(model, topk=10):
    topic_words = [[word for word, weight in model.get_topic(tid)[:topk] if word and weight > 0]
                   for tid in model.get_topics() if tid != -1]
    terms = [word for topic in topic_words for word in topic]
    if not terms:
        return {"coherence_c_v": np.nan, "topic_diversity": np.nan, "coherence_status": "no non-noise terms"}
    assert all(word in coherence_dictionary.token2id for word in terms), "Coherence vocabulary mismatch"
    valid_topics = [words for words in topic_words if len(words) >= 2]
    coherence = (CoherenceModel(topics=valid_topics, texts=tokenized_corpus,
                               dictionary=coherence_dictionary, coherence="c_v", window_size=110,
                               processes=1).get_coherence() if valid_topics else np.nan)
    status = "ok" if np.isfinite(coherence) and len(valid_topics) == len(topic_words) else "undefined or partial; inspect topics"
    return {"coherence_c_v": coherence, "topic_diversity": len(set(terms)) / len(terms),
            "coherence_status": status, "coherence_topics_scored": len(valid_topics)}

score_table = pd.DataFrame([{"model": name, **topic_scores(model)} for name, model in
                           [("PCA + HDBSCAN", pca_topic_model), ("UMAP + HDBSCAN", umap_topic_model)]])
comparison = comparison.drop(columns=[c for c in score_table if c != "model"], errors="ignore").merge(score_table, on="model", validate="one_to_one")
display(comparison.round(3))


## 🔎 4.10 Stabilność UMAP względem seed

Używamy seed 1, 7, 21, 42 i 99. Porównujemy liczbę tematów, szum i ARI przypisań. `ARI_all` może rosnąć, gdy wiele tekstów pozostaje szumem. Dlatego obok pokazujemy ARI na wspólnych inliers oraz ich udział.

Nie przyjęliśmy progu pozwalającego nazwać model stabilnym. Opisujemy większą lub mniejszą zgodność, a jeden dominujący klaster nie stanowi dowodu użyteczności nawet przy wysokim ARI.

### ▶️ Krok kodu 16

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
STABILITY_SEEDS = [1, 7, 21, 42, 99]

# For workshop speed, reuse the cached sentence embeddings.
# Each seed changes UMAP but keeps HDBSCAN and all topic-representation settings fixed.

def fit_umap_assignment(seed):
    reducer = UMAP(
        n_neighbors=15,
        n_components=N_COMPONENTS,
        min_dist=0.0,
        metric="cosine",
        random_state=seed,
        n_jobs=1,
    )
    reduced = reducer.fit_transform(embeddings)
    clusterer = HDBSCAN(**HDBSCAN_PARAMS)
    return clusterer.fit_predict(reduced)

seed_assignments = {seed: (np.asarray(topics_umap) if seed == RANDOM_STATE else fit_umap_assignment(seed))
                    for seed in STABILITY_SEEDS}

stability_rows = []
for a, b in combinations(STABILITY_SEEDS, 2):
    la, lb = seed_assignments[a], seed_assignments[b]
    common = (la != -1) & (lb != -1)
    stability_rows.append({
        "ARI_common_inliers": adjusted_rand_score(la[common], lb[common]) if common.sum() >= 2 else np.nan,
        "common_inlier_share": np.mean(common),
        "seed_a": a,
        "seed_b": b,
        "ARI_all": adjusted_rand_score(la, lb),
        "topics_a": len(set(la)) - (1 if -1 in la else 0),
        "topics_b": len(set(lb)) - (1 if -1 in lb else 0),
        "outliers_a": np.mean(la == -1),
        "outliers_b": np.mean(lb == -1),
    })

stability_df = pd.DataFrame(stability_rows)
display(stability_df.round(3))
print("Mean pairwise ARI:", round(stability_df["ARI_all"].mean(), 3))

seed_summary = pd.DataFrame([{"seed":seed,
    **summarize_assignments("UMAP", labels, y_external)} for seed,labels in seed_assignments.items()])
display(seed_summary[["seed","n_topics","outlier_rate","largest_topic_share_inliers"]].round(4))


## 🔎 4.11 Sensitivity analysis HDBSCAN

Zachowujemy siatkę `min_cluster_size = 15, 30, 60`. Najpierw grupujemy te same zredukowane embeddingi. Dla każdego ustawienia wyznaczamy też reprezentację BERTopic przy wspólnych parametrach i sprawdzamy zgodność przypisań z surowym HDBSCAN.

Tabela obejmuje pokrycie, koncentrację, coherence, diversity i kryteria zewnętrzne. Nie wybieramy ustawienia według jednej liczby. Mniejszy próg może zwiększać szczegółowość, a większy może dawać szersze tematy, ale nie są to gwarancje.

### ▶️ Krok kodu 17

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
MIN_CLUSTER_SIZES = [15, 30, 60]

sensitivity_rows = []
sensitivity_assignments = {}
for reducer_name, reduced in [("PCA", pca_10), ("UMAP", umap_10)]:
    for mcs in MIN_CLUSTER_SIZES:
        clusterer = HDBSCAN(**{**HDBSCAN_PARAMS, "min_cluster_size": mcs})
        labels = clusterer.fit_predict(reduced)
        inlier = labels != -1
        counts = np.unique(labels[inlier], return_counts=True)[1]
        sensitivity_assignments[f"{reducer_name}_{mcs}"] = labels
        if mcs == HDBSCAN_PARAMS["min_cluster_size"]:
            fitted = pca_topic_model if reducer_name == "PCA" else umap_topic_model
        else:
            reducer = (PCA(n_components=N_COMPONENTS, svd_solver="full") if reducer_name == "PCA"
                       else UMAP(n_neighbors=15, n_components=N_COMPONENTS, min_dist=0.0,
                                 metric="cosine", random_state=RANDOM_STATE, n_jobs=1))
            fitted = make_topic_model(reducer)
            fitted.hdbscan_model = HDBSCAN(**{**HDBSCAN_PARAMS, "min_cluster_size": mcs})
            fitted_topics, _ = fitted.fit_transform(representation_texts, embeddings=embeddings)
            assert adjusted_rand_score(labels, fitted_topics) == 1.0
        quality = topic_scores(fitted)
        sensitivity_rows.append({
            "reducer": reducer_name,
            "min_cluster_size": mcs,
            "n_topics": len(set(labels)) - (1 if -1 in labels else 0),
            "outlier_rate": np.mean(~inlier),
            "largest_topic_share_inliers": counts.max()/counts.sum() if len(counts) else np.nan,
            **quality,
            "NMI_all": normalized_mutual_info_score(y_external, labels),
            "ARI_all": adjusted_rand_score(y_external, labels),
        })

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))


# 🔎 5. Wykresy
## 🔎 5.1 Szum i rozmiar klastra

Porównujemy odsetki szumu dla tej samej, wcześniej określonej siatki. Obok należy czytać tabelę koncentracji i jakości reprezentacji.

### ▶️ Krok kodu 18

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for reducer_name, group in sensitivity.groupby("reducer"):
    ax.plot(group["min_cluster_size"], group["outlier_rate"], marker="o", label=reducer_name)
ax.set_xlabel("HDBSCAN min_cluster_size")
ax.set_ylabel("Odsetek szumu")
ax.set_title("Szum a min_cluster_size")
ax.legend()
(BASE_DIR / "outputs").mkdir(exist_ok=True)
fig.tight_layout()
fig.savefig(BASE_DIR / "outputs/sensitivity_plot.png")
plt.show()


## 🔎 5.2 Projekcje 2D

Wykresy pomagają zobaczyć reprezentację. Nie są dowodem przewagi modelu. Do klasteryzacji używaliśmy 10 wymiarów, a poniżej tworzymy osobne projekcje dwuwymiarowe. Pozorna separacja może wynikać z redukcji.

### ▶️ Krok kodu 19

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
pca_2 = PCA(n_components=2, svd_solver="full").fit_transform(embeddings)
umap_2 = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    n_jobs=1,
).fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pca_2[:, 0], pca_2[:, 1], s=7, alpha=.45)
ax.set_title("Projekcja PCA embeddingów MiniLM")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(umap_2[:, 0], umap_2[:, 1], s=7, alpha=.45)
ax.set_title("Projekcja UMAP embeddingów MiniLM")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.show()


# 🔎 6. Interpretacja
## 🔎 Zapis wyników

Eksport obejmuje podsumowania modeli, przypisania dokumentów, reprezentantów i analizę wrażliwości. Pozwala wrócić od każdej liczby do tekstu i ustawień analizy.

### ▶️ Krok kodu 20

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
OUTPUT_DIR = BASE_DIR / "outputs" / ("fast" if FAST_MODE else "full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(OUTPUT_DIR / "pca_vs_umap_model_comparison.csv", index=False)
sensitivity.to_csv(OUTPUT_DIR / "hdbscan_sensitivity.csv", index=False)
stability_df.to_csv(OUTPUT_DIR / "umap_seed_stability.csv", index=False)

assignments = work_df[["source_row", "target", "document"]].copy()
assignments["topic_pca"] = topics_pca
assignments["topic_umap"] = topics_umap
assignments.to_csv(OUTPUT_DIR / "document_topic_assignments.csv", index=False)

pca_topic_model.get_topic_info().to_csv(OUTPUT_DIR / "topic_info_pca.csv", index=False)
umap_topic_model.get_topic_info().to_csv(OUTPUT_DIR / "topic_info_umap.csv", index=False)

print("✅ Zapisano w:", OUTPUT_DIR.resolve())

representatives.to_csv(OUTPUT_DIR / "representative_documents.csv", index=False)
if topic_barchart is not None:
    topic_barchart.write_html(OUTPUT_DIR / "topic_barchart.html", include_plotlyjs=True)
seed_frame = pd.DataFrame({str(seed): labels for seed, labels in seed_assignments.items()})
seed_frame.insert(0, "source_row", work_df["source_row"])
seed_frame.to_csv(OUTPUT_DIR / "umap_seed_assignments.csv", index=False)
packages = ["numpy", "pandas", "scipy", "scikit-learn", "sentence-transformers", "transformers",
            "torch", "umap-learn", "hdbscan", "bertopic", "spacy", "gensim", "matplotlib", "plotly"]
metadata = {"runtime_seconds": time.perf_counter() - RUN_STARTED, "fast_mode": FAST_MODE,
            "python": platform.python_version(), "packages": {p: version(p) for p in packages},
            "input_file": DATA_PATH.name, "input_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
            "documents": len(texts), "empty_representations": empty_representations,
            "cache_key": cache_key, "cache_hit": cache_hit,
            "embedding_sha256": embedding_digest, "embedding_config": cache_config,
            "spacy_model": nlp.meta.get("version"), "hdbscan": HDBSCAN_PARAMS,
            "pca": {"n_components": N_COMPONENTS, "svd_solver": "full"},
            "umap": {"n_neighbors": 15, "n_components": N_COMPONENTS, "min_dist": 0.0,
                     "metric": "cosine", "random_state": RANDOM_STATE, "n_jobs": 1},
            "seeds": STABILITY_SEEDS, "min_cluster_sizes": MIN_CLUSTER_SIZES,
            "vectorizer": {"ngram_range": [1, 2], "vocabulary_document_min_df": 2},
            "mmr_diversity": 0.1, "coherence": "gensim c_v; ordered unigram/bigram features; window_size=110"}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Czas obliczeń (sekundy):", round(metadata["runtime_seconds"], 1))

seed_summary.to_csv(OUTPUT_DIR / "umap_seed_summary.csv", index=False)
sensitivity_frame = pd.DataFrame(sensitivity_assignments)
sensitivity_frame.insert(0, "source_row", work_df.source_row)
sensitivity_frame.to_csv(OUTPUT_DIR / "sensitivity_assignments.csv", index=False)


## 🔎 Interpretacja rzeczywistego rozwiązania

W wykonaniu referencyjnym przedstawionym na slajdach PCA odrzuca jako szum dużą część korpusu. UMAP skupia prawie wszystkich inliers w jednym temacie. Zgodność z target jest słaba w obu przypadkach. **Żadna konfiguracja domyślna nie powinna automatycznie stać się końcowym rozwiązaniem merytorycznym.**

Dalsza praca obejmuje analizę parametrów HDBSCAN, seed i dokumentów reprezentatywnych oraz granicznych. Można ponownie rozważyć liczbę zredukowanych wymiarów lub założenia klasteryzacji, dokumentując tę decyzję. W tym warsztacie nie dostrajamy modelu do target. Wyniki liczbowe odczytujemy z aktualnej tabeli powyżej.

⚠️ W nowej sesji liczby klastrów mogą się różnić między platformami mimo tego samego seed. Przeczytaj własne tabele powyżej: ten akapit opisuje wykonanie referencyjne, a nie gwarantowany rezultat każdej sesji.

**Porównanie wykonanych sesji:** w wykonaniu Windows PCA dało 5 tematów, a UMAP 2. W sprawdzonym Colabie CPU PCA dało 5 tematów (86,9% szumu), a UMAP 35 (35,6% szumu). To pokazuje, dlaczego oceniamy własne tabele, stabilność i treść dokumentów zamiast oczekiwać konkretnej liczby grup.

# 🔎 7. Ćwiczenie

Wybierz ten sam `source_row` w wynikach słownika, VADER i BERTopic. Opisz, co wnosi każda metoda. Przeczytaj reprezentanta, tekst graniczny i dokument ze szumu przed nazwaniem tematu.

Sygnał ilościowy → powrót do tekstu → interpretacja jakościowa → interpretacja teoretyczna → sprawdzenie odporności.

Zapisz, jak lektura zmieniła wniosek. Samo połączenie kilku metod ilościowych nie wystarcza do pełnego projektu mixed methods. Potrzebny jest jawny sposób integracji jakościowej i ilościowej.

## 💾 Pobranie wyników

Uruchom tę komórkę po ukończeniu analizy. Utworzy ZIP i w Colab rozpocznie pobieranie. Jeśli przeglądarka je zablokuje, odszukaj ZIP w panelu Pliki i pobierz ręcznie. Zachowaj też własną kopię notebooka.

In [ ]:
import shutil
bundle = Path('wyniki_modul_04')
bundle.mkdir(exist_ok=True)
if not Path(OUTPUT_DIR).exists():
    raise RuntimeError('⛔ Najpierw wykonaj komórki analizy i zapisu wyników.')
shutil.copytree(OUTPUT_DIR, bundle / 'tabele_i_wykresy', dirs_exist_ok=True)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('✅ Plik wyników:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
